# Addestramento YOLO: 4 oggetti della demo (dataset 1000 + 500)

Classi: `tappo_bianco`, `anello_verde`, `vite_arancione`, `secchio_blu`.

Prima: *Runtime -> Cambia tipo di runtime -> GPU*. Poi esegui le celle in ordine, **una volta sola ciascuna**.

Cosa non andava nella versione di prima (per questo i modelli scaricati erano identici al primo):
1. lo zip da 1000 ha `images/` e `labels/` nella radice, ma l'unione cercava `dataset/`: **le 1070 immagini nuove
   non venivano mai copiate**, senza errori;
2. valutazione ed export caricavano sempre `runs/oggetti` (il primo addestramento), non quello appena fatto.

Ora: l'unione trova le immagini in qualunque struttura (`images/train`, `dataset/images/train`, `train/images`),
rimappa le classi **per nome**, toglie i doppioni (le 18 foto vere sono in tutti e due gli zip) e si ferma con un
errore se manca qualcosa; un solo addestramento; i file scaricati hanno il nome dell'addestramento.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics
# Solo se l'addestramento si ferma con un errore di triton/torch:
# !pip install -q triton==3.6.0

## 1. Carica i dataset
Seleziona `dataset_oggetti_1000.zip` e, se lo hai, anche `dataset_oggetti.zip` (500): le immagini diverse si
aggiungono, i doppioni si scartano. Se sono già in `/content` salta la cella.
Facoltativo: carica il modello di prima rinominato `oggetti_best_vecchio.pt` per vedere il confronto alla fine.

In [ ]:
from google.colab import files
files.upload()
!ls -la /content/*.zip /content/*.pt 2>/dev/null

## 2. Unisci i dataset
Per aggiungere foto vere della D435 etichettate: mettile in uno zip con `images/train` e `labels/train` e aggiungilo
a `ZIPS` con una sigla nuova. Le immagini senza etichetta restano come **sfondi vuoti** (aiutano contro i falsi
positivi: più sfondi del laboratorio ci sono, meglio è).

In [ ]:
import hashlib
import os
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import yaml

CLASSES = ['tappo_bianco', 'anello_verde', 'vite_arancione', 'secchio_blu']  # ordine = id del modello
ZIPS = {'d1000': '/content/dataset_oggetti_1000.zip', 'd500': '/content/dataset_oggetti.zip'}  # il primo vince sui doppioni
OUT = Path('/content/dataset_combinato')
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLITS = {'train': 'train', 'training': 'train', 'val': 'val', 'valid': 'val', 'validation': 'val', 'test': 'test'}


def split_of(rel_dir):
    """train/val/test dalla cartella più vicina al file (images/train/x.jpg e train/images/x.jpg)."""
    return next((SPLITS[p.lower()] for p in reversed(rel_dir.parts) if p.lower() in SPLITS), None)


def label_of(img):
    """.../images/<split>/x.jpg -> .../labels/<split>/x.txt (anche train/images -> train/labels)."""
    parts = list(img.parts)
    if 'images' in parts:
        i = len(parts) - 1 - parts[::-1].index('images')
        parts[i] = 'labels'
    return Path(*parts).with_suffix('.txt')


def class_map(root):
    """Id del data.yaml dello zip -> id nostri, per nome. Senza data.yaml: stesso ordine di CLASSES."""
    ys = sorted(Path(root).rglob('data.yaml'))
    if not ys:
        return {i: i for i in range(len(CLASSES))}
    names = yaml.safe_load(ys[0].read_text())['names']
    names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
    unknown = [n for n in names if n not in CLASSES]
    if unknown:
        raise ValueError(f'{ys[0]}: classi {unknown} non previste. Attese: {CLASSES}')
    return {i: CLASSES.index(n) for i, n in enumerate(names)}


def build_dataset(zips, out):
    shutil.rmtree(out, ignore_errors=True)
    count, boxes, seen, present = Counter(), Counter(), set(), set()
    for tag, zp in zips.items():
        if not os.path.exists(zp):
            print(f'MANCA {zp}: lo salto')
            continue
        tmp = out.parent / f'_tmp_{tag}'
        shutil.rmtree(tmp, ignore_errors=True)
        zipfile.ZipFile(zp).extractall(tmp)
        cmap = class_map(tmp)
        imgs = sorted(p for p in tmp.rglob('*') if p.suffix.lower() in IMG_EXT and '__MACOSX' not in p.parts)
        for img in imgs:
            split = split_of(img.relative_to(tmp).parent)
            if split is None:
                count[f'{tag} fuori da train/val/test (saltate)'] += 1
                continue
            digest = hashlib.md5(img.read_bytes()).hexdigest()
            if digest in seen:  # stessa foto già presa da un altro zip
                count[f'{tag} {split} doppioni scartati'] += 1
                continue
            seen.add(digest)
            lines = []
            lab = label_of(img)
            if lab.exists():
                for row in lab.read_text().splitlines():
                    v = row.split()
                    if v:
                        v[0] = str(cmap[int(v[0])])
                        boxes[split, CLASSES[int(v[0])]] += 1
                        lines.append(' '.join(v))
            dst = out / 'images' / split / f'{tag}_{digest[:8]}_{img.stem}{img.suffix.lower()}'
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img, dst)
            dst_lab = out / 'labels' / split / f'{dst.stem}.txt'
            dst_lab.parent.mkdir(parents=True, exist_ok=True)
            dst_lab.write_text('\n'.join(lines) + ('\n' if lines else ''))
            present.add(split)
            count[f'{tag} {split}'] += 1
            count[f'{tag} {split} di cui sfondi vuoti'] += not lines
        shutil.rmtree(tmp)

    if not {'train', 'val'} <= present:
        raise RuntimeError(f'Servono immagini di train e di val, trovati solo: {sorted(present)}')
    data = {'path': str(out), 'train': 'images/train', 'val': 'images/val', 'names': dict(enumerate(CLASSES))}
    if 'test' in present:
        data['test'] = 'images/test'
    (out / 'data.yaml').write_text(yaml.safe_dump(data, sort_keys=False))

    print('Immagini:')
    for k in sorted(count):
        print(f'  {k:42s} {count[k]}')
    print('Riquadri per classe:')
    for s in ('train', 'val', 'test'):
        if s in present:
            print(f'  {s:5s} ' + '  '.join(f'{c}={boxes[s, c]}' for c in CLASSES))
    return count


if __name__ == '__main__':
    build_dataset(ZIPS, OUT)


## 3. Addestra
Circa 20-40 minuti su L4/A100. `RUN` è la cartella vera di questo addestramento: le celle dopo usano quella.
Per partire dal modello di prima invece che da zero: carica `oggetti_best_vecchio.pt` e metti `START` a quel file
(più veloce; da zero sul dataset completo di solito generalizza meglio).

In [ ]:
from ultralytics import YOLO

START = 'yolo11s.pt'  # pesi COCO di partenza; 'yolo11n.pt' = più veloce sul PC, un po' meno preciso
model = YOLO(START)
model.train(
    data=str(OUT / 'data.yaml'), imgsz=640, epochs=150, patience=40, batch=32,
    # Oggetti visti dall'alto: qualsiasi rotazione e ribaltamento sono realistici
    degrees=180, flipud=0.5, fliplr=0.5,
    # Le classi si distinguono per colore: tinta quasi ferma, luce e saturazione libere
    hsv_h=0.01, hsv_s=0.5, hsv_v=0.4,
    scale=0.5, translate=0.15, mosaic=1.0, mixup=0.1, close_mosaic=15,
    project='/content/runs', name='oggetti_1000', exist_ok=False,  # cartella nuova a ogni run
)
RUN = Path(model.trainer.save_dir)
BEST = RUN / 'weights' / 'best.pt'
print('Pesi di questo addestramento:', BEST)

## 4. Valuta sulle 18 foto vere
Le foto vere non sono mai usate in training: questo è il numero che conta. Con poche foto (l'anello verde compare
solo 3 volte) i valori per classe oscillano, guardali insieme alle immagini qui sotto.

In [ ]:
import glob

if 'BEST' not in globals():  # runtime riavviato: prende l'addestramento più recente
    BEST = Path(max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime))
    RUN = BEST.parent.parent
print('Valuto:', BEST)

models = {'nuovo': BEST}
if os.path.exists('/content/oggetti_best_vecchio.pt'):
    models['vecchio'] = '/content/oggetti_best_vecchio.pt'

for label, weights in models.items():
    m = YOLO(str(weights))
    r = m.val(data=str(OUT / 'data.yaml'), imgsz=640, split='val', plots=False, verbose=False)
    per_class = '  '.join(f'{m.names[c]}={r.box.ap50[i]:.2f}' for i, c in enumerate(r.box.ap_class_index))
    print(f'{label:7s} mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}   {per_class}')

In [ ]:
from IPython.display import Image, display

YOLO(str(BEST)).predict(sorted(glob.glob(str(OUT / 'images/val/*'))), imgsz=640, conf=0.4, save=True,
                        project='/content/runs', name='pred', exist_ok=True, verbose=False)
for f in sorted(glob.glob('/content/runs/pred/*.jpg'))[:8]:
    display(Image(f, width=500))

## 5. Esporta e scarica
Nel progetto il file va copiato sopra `data/models/oggetti_best.onnx`.

In [ ]:
import torch
from google.colab import files

onnx_path = YOLO(str(BEST)).export(format='onnx', imgsz=640, opset=12, simplify=True)
name = f'oggetti_best_{RUN.name}'
shutil.copy(BEST, f'/content/{name}.pt')
shutil.copy(onnx_path, f'/content/{name}.onnx')

ck = torch.load(BEST, map_location='cpu', weights_only=False)
print('Addestramento :', RUN)
print('Salvato il    :', ck.get('date'))
print('Immagini train:', len(list((OUT / 'images/train').iterdir())))
print('sha256 onnx   :', hashlib.sha256(open(f'/content/{name}.onnx', 'rb').read()).hexdigest()[:16])
files.download(f'/content/{name}.pt')
files.download(f'/content/{name}.onnx')